# 고급 RAG

고급 RAG(Advanced RAG)를 다룹니다. 더 나은 결과를 위해 RAG에서 성능을 더 뽑아내는 기법들, 전처리(pre-precessing), 리랭킹(re-ranking) 등을 직접 실험해보고, 마지막엔 강력한 성능의 고급 RAG 파이프라인을 완성하게 됩니다.

## 복습 : RAG는 어떻게 동작하는가?
1. 사용자의 질문을 LLM에 바로 넘기는게 아니라
2. 임베딩 모델(인코더)로 질문을 벡터화
3. 지식 저장소에서 그 벡터와 유사한 벡터를 가진 청크를 찾음(문서를 미리 청크로 나눠서 인덱싱 해둔 것)
4. 가장 관련성 높은 청크를 검색
5. 그 청크들을 프롬프트에 넣어서 LLM에 전달
6. 답변이 생성됨

## 복습 : 평가(evaluation) 개념
RAG 작업은 부두교(voodoo) 처럼 느껴질 수 있고, 추측이 많이 필요하고 손이 많이 가는 작업처럼 느껴지는데 이를 과학적으로/방탄으로 만드는 방법은 강력한 지표를 세우고 실험을 반복하는것 입니다. 

1단계 : 골든 데이터셋 큐레이션 (질문 + 참고 답변 세트)
2단계 : 검색(retrieval) 지표
    * MRR : 정답 청크가 처음 몇 번째에 나왔는지의 역수를 평균. 항상 1위면 MRR=1
    * NDCG : 정답 청크들이 결과 전체에 얼마나 잘 분산(상위에 몰려있는지)되어 있는지, 로그를 이용해 아래쪽에 있는 히트에 약한 패널티를 주고 최고 점수로 나눠서 정규화(그래서 완벽하면 항상 1)
    * Recall(재현율) : 상위 K개 청크 중 정답이 포함된 테스트 케이스의 비율. 예) 100개 테스트 중 50개가 상위 3개 안에 정답을 포함 -> Recall@3 = 50%
    * Precision(정밀도) : 노출된 청크 중 실제 관련 있는 청크의 비율

Recall vs Precision 차이 : Recall은 "검색을 시도한 것 중 몇 %가 성공했는가?"(테스트 케이스 기준), Precision은 "노출한 것 중 몇 %가 실제로 유용했는가"(청크 기준).
RAG에서는 Recall이 더 중요. 관련 없는 내용이 좀 섞여도 LLM이 대체로 잘 무시하지만, 애초에 정답이 검색되지 않으면 답을 낼 수 없기 때문. 키워드 커버리지도 이런 Recall류 지표에 해당
3단계 : 답변 품질 지표 - LLM-as-judge가 특히 유용


# 고급 RAG 기법
몇년전 RAG가 인기를 끌면서 "RAG를 더 잘하는 법"을 둘러싼 수많은 용어와 기법들이 생겨났는데 이걸 유사과학(pseudo-science) 이라고 부릅니다. 이 용어들은 결국 "이것저것 시도해보다가 뭔가 잘 안 될 때 문제를 해결하려고 다르게 시도해본것"들을 그럴싸하게 공식화한 것 일 뿐입니다. 
여기서 "어떤 RAG 기법을 써야 하나요?" 라고 질문을 한다면 이건 아주 나쁜 질문 입니다. 이유는 마법 같은 정답은 없기 때문. "이 인코더를 써라", "리랭킹을 써라"라고 일반화해서 말할 수 없고, 정답은 지표를 세우고, 평가를 만들고, 직접 실험 해보는 것 뿐입니다. 지름길은 없습니다.

이건 실제 엔지니어링이 아니라, 비즈니스 케이스와 데이터셋에 따라 다른 기법이 통할 수 있는 실험의 영역이라는 것. 그래서 10가지 기법을 소개하지만 "이게 정답"이라는 식으로 받아들이지 말고 직접 검증해봐야 합니다.

## 기법 1 : 청킹(Chunking)
LangChain의 다양한 텍스트 스플리터(청커)로 파라미터를 바꿔가며 실험. 단순한 "시행착오"이지만 무작정 추측이 아니라 정보에 기반한 시행착오가 되어야함 즉:
* 잘못된 답변을 유발하는 특정 문제를 발견
* 그 문제를 해결하기 위한 구체적인 가설을 세우고 청킹을 변경
* 측정해서 효과를 확인
(어느 정도 탐색적 R&D도 괜찮지만, 기본은 "문제 -> 가설 -> 변경 -> 측정"의 사이클)

## 기법 2 : 인코더(Encoder) 선택
테스트셋과 비즈니스 목표에 맞는 최적의 인코더 모델을 고르는 것.

### 이미지가 포함된 지식창고 처리:
* 옵션 A : 이미지와 텍스트를 통일한 벡터 공간에 매핑할 수 있는 멀티모달 인코더 사용(sentence Transformers 허깅페이스에 이런 모델들 있음) (살짝 불안정함)
* 옵션 B : 먼저 모델로 이미지에 질문과 잘 매칭될 캡션을 생성하고, 그 텍스트를 벡터화하는 방식. 이게 더 안정적으로 매칭될 가능성이 높다고 봄
꼭 직접 사용 사례에 맞게 테스트 해볼 것

### PDF/Word 문서 벡터화 관련 팁:
* PDF를 직접 인코딩하는 모델도 있지만, PDF는 바이너리 객체라 벡터화에 적합하지 않으니 추천 하지 않음
* 대신 PDF -> 마크다운 변환 후 벡터화 추천 (모델들이 마크다운으로 많이 학습되어 있음)
* 단, 이 변환 작업에 LLM을 쓰지 말것 - "LLM은 끔찍한 오용" PDS 파싱은 이미 해결된 문ㅁ제이니 파이썬 라이브러리 로 처리하면 충분함. AI에 모든걸 맡기지 말고 적절한 도구를 써야함
* 원칙 : 형식 변환은 소프트웨어로, LLM에게 잘 맞는 형식으로 변환한 뒤 벡터화

## 기법 3: 프롬프트 작업 (당연해 보이지만 중요)
고급 기법이라고 부르기엔 너무 기본적으로 들리지만 실제로 큰 차이를 만듦:
* RAG로 가져온 관련 컨텍스트 외에도, 모든 프롬프트에 넣을 수 있는 고정된 배경 정보가 있는 경우가 많음
* 토큰 몇백 개 정도 낭비하는 건 큰 비용이 아니므로, 좋은 맥락을 위해서라면 아깝지 않음
* 예) 현재 날짜를 프롬프트에 항상 넣는 것 같은 표준 팁
* 사용자 발화, 관련 컨텍스트, 대화 히스토리를 잘 이해하고 처리하도록 프롬프트를 다듬는 것만 으로도 큰 개선 효과

## 기법 4 : 문서 사전 처리 - 진짜 고급 기법
문서를 벡터 스터어에 넣기 전에 LLM으로 먼저 재작성하는 기법

예시) 항공권 가격이 담긴 숫자 표 - 이걸 그대로 벡터화 하면 그냥 "숫자로 된 표"라는 의미밖에 안 담겨서 검색에 별로 유용하지 않음. 대신 LLM에게 "이 지식창고의 섹션을 쿼리에 가장 적합한 형식으로 다시 써달라"고 요청한 뒤, 그 결과를 벡터화하면 훨씬 검색에 잘 맞는 텍스트가 됨.

시멘틱 청킹 과도 연결됨: 무작위로 텍스트를 자르는 게 아니라, 의미 단위(한 사람의 경력 단계별, 제품의 각 측면별)로 나누는 청킹 방식. 랭체인에 실험적 패키지로 시맨틱 청커가 있긴 하지만, 더 쉬운 방법은 문서 전처리 단계에서 LLM에게 "의미 있는 단위로 나누고, 지식창고에 적합하게 재작성해달라"고 한번에 요청하는 것 - 즉 "기법1 + 기법4"를 합친 방식

## 기법 5 : 쿼리 재작성 (Query Rewriting) - 문서 전처리의 반대편
문서를 다루는게 아니라 사용자의 질문 자체를 다루는 기법. 사용자의 질문이:
* 며칠 전 논의의 요점만 담고 있거나
* 이전 질문에 대한 후속 질문(follow-up)이라

RAG 검색에 그대로 쓰기엔 적합하지 않을 수 있습니다. 코드로 직접 이런 맥락 결합을 처리하는건 매우 어렵지만, LLM에게 맡기면 쉽게 처리 됩니다. 대화 히스토리를 함께 전달해서 "이 질문을 지식창고 조회에 가장 적합한 형태로 다시 써줘"라고 요청하는 방식

-> 문서 전처리 + 쿼리 재작성 은 강력한 조합으로 함께 쓰이는 경우가 많습니다.

## 기법 6 : 쿼리 확장
매우 흔하게 쓰이는 기법. 쿼리 재작성의 확장판이라고 볼 수 있는데, 모델에게 쿼리를 하나만 만들게 하지 말고, 여러 개의 서로 다른 쿼리를 만들게 해서 각각으로 데이터베이스 조회를 수행하는 것. 예를 들어 다른 RAG쿼리 3개를 만들고 각각에 대해 유사 문서를 수집.

## 기법 7 : 리랭킹
특히 쿼리 확장을 쓰면 검색되는 청크 수가 많아짐. 이렇게 모인 청크들은 질문과의 관련성 순서로 재배열 하는 것이 리랭킹.

방법 : LLM에게 "너는 답변할 책임이 없다. 다른 LLM이 답변을 만들 거다. 너는 그냥 관련성이 높은 것 부터 낮은 것 순으로 정렬만 해달라"고 요청. 정렬 후 관련성 낮은 것들은 잘라내서 실제 답변 생성용 LLM에는 최종 관련성 높은 청킄만 전달.

특히 앞 단계들(쿼리 확장 등)을 거쳐 청크가 많아진 경우, 컨텍스트를 관련 없는 청크로 오염시키지 않기 위해 리랭킹이 매우 중요해질 수 있음

## 기법 8 : 계층적 RAG(Hierarchical RAG)
RAG의 대표적인 약점 - 어려 문서에 걸친 질문에 잘 답하지 못하는 문제를 해결하려는 기법.
아이디어 : 지식창고를 여러 레벨의 요약(롤업)으로 계층 구조화. 예) 모든 제품에 대한 요약, 모든 직원에 대한 요약 문서를 만들어 둠. RAG 조회 시 먼저 요약본에서 대략적인 정보를 얻고, 필요하면 세부 청크로 드릴다운.

예시 : 연봉이 X 미만인 직원이 몇 명인가? -> 개별 직원 문서를 전체를 뒤지는 대신, 모든 직원 +  연봉을 압축한 요약 문서 하나로 답할 수 있음. 이런 유형의 질문은 지식창고 전체에 걸친 정보가 필요하기 때문에 요약 없이는 답하기 어려움

한계 : 요약을 어떤 방식으로 롤업할지 미리 정해놔야 하는데, 사용자가 전혀 다른 축으로 질문하면 ("이름이 A로 시작하는 문서가 몇 개인가?" - 알파벳 순으로 계층을 안 만들어 놨다면) 답을 못 찾음. 결국 또 두더지 잡기. 새로운 유형의 문서 간 질문이 나타날 때마다 새로운 계층적 요약 문서를 만들어 대응하는 식.

## 기법 9: 그래프 RAG(Graph RAG)

이름은 거창하게 들리지만 아이디어는 단순함: 지식창고의 문서들이 서로 고립되어 있지 않고 관계를 가지는 경우가 많다는 것. (예: 직원에게 상사가 있고, 그 상사에게도 또 상사가 있는 것처럼)

메타데이터 기반 방식: 각 청크에 메타데이터를 붙여서 "이 청크는 같은 문서의 다른 청크와 관련" 또는 "상사 관련 청크"처럼 관계를 기록해두면, 조회 시 그 메타데이터를 따라 1홉 또는 2홉 떨어진 관련 청크까지 컨텍스트에 함께 넣을 수 있음.

그래프 데이터베이스 방식: Neo4j 같은 그래프 DB는 정보를 **노드(entity)와 엣지(관계)**로 저장하도록 설계되어 있어서, 벡터로 가까운 청크를 찾은 뒤 그 청크의 그래프상 1~2홉 이웃 청크까지 쉽게 포함시킬 수 있음.

트렌디하지만 관계가 매우 많은 매우 특정한 상황에서만 잘 작동하는 편. 대부분의 경우엔 메타데이터만으로도 충분히 해결 가능함. 그래프 DB가 꼭 필요한 특정 문제가 있는 게 아니라면 그렇게까지 필요하지 않지만, 지켜보고 실험해볼 만한 영역.

## 기법 10: 에이전틱 RAG(Agentic RAG)

기존 RAG는 매우 선형적(linear): 질문 → 벡터 조회 → 컨텍스트 삽입 → LLM 호출.

2024년("고대사")식 접근과 다르게, 요즘 방식은 LLM이 스스로 결정하게 하는 것: 벡터 조회 자체를 LLM이 사용할 수 있는 **도구(tool)**로 제공하고, 거기에 SQL 실행 도구, 파일 문자열 검색 도구, API 호출 도구 등 여러 도구를 함께 제공. 그러면 이 LLM(에이전트)이 사용자 질문을 받고 스스로 "벡터 조회를 할지, SQL을 실행할지, API를 부를지"를 판단해서 필요한 정보를 모아 답변.

이는 사실 앞의 여러 기법(쿼리 확장, 쿼리 재작성, 리랭킹)을 LLM이 스스로 판단해서 어떤 순서로 적용할지 결정하는 것과 비슷함. 즉 앞의 9가지 기법을 "LLM이 알아서 오케스트레이션"하는 것. 결과가 안 나오면 계속 시도하는 루프(loop) 구조가 될 수도 있음.

장단점:

장점: 매우 강력함, 이전엔 불가능했던 답변까지 얻을 수 있음. 모델이 만족할 때까지 도구 호출을 반복
단점: 예측 불가능성 증가. 오늘 좋은 결과가 나왔다고 내일 같은 질문에 똑같이 좋은 결과가 나온다는 보장이 없음 - 재현성(repeatability), 견고성(robustness) 측면에서 자율적 접근 방식의 흔한 단점

핵심: 에이전틱 RAG도 여전히 앞서 나온 단계들 대부분을 수행하지만, 미리 정해진 순서(hard-coded orchestration)가 아니라 도구 호출(tool call)을 통해 유연하게 수행한다는 점이 다를 뿐.

## 마무리: "RAG는 죽었다"는 밈에 대한 반박

흔히 듣는 두 가지 "RAG는 죽었다"는 주장:

주장 1: "컨텍스트 창이 이제 엄청 커져서, 지식창고 전체를 그냥 넣고 트랜스포머의 attention이 알아서 관련 있는 걸 찾게 하면 되지 않나?"
→ 반박: 말도 안 되는 소리. 8주차 예시처럼 지식창고가 극단적으로 커지는 경우도 있고, 그걸 통째로 LLM에 넣거나 쪼개서 여러 번 호출하는 건 시간 효율이 매우 나쁨. 벡터 기반 검색 등으로 관련 없는 90%를 쉽게 버릴 수 있는데, 컨텍스트 창이 얼마나 커지든 관련 없는 내용을 걸러내는 접근법은 항상 중요할 것이라는 게 강사 입장.

주장 2: "에이전트가 등장했으니, 벡터 조회 후 관련 컨텍스트를 보여주는 파이프라인 방식(기법 10 이전의 방식들)은 이제 구식이다. 에이전트가 스스로 데이터를 파헤치는 방법을 알아서 결정하게 하면 된다."
→ 반박: 흥미로운 지적이지만, 이건 그저 다른 이름의 RAG일 뿐이라는 것. "에이전틱"이라고 부르면 세련되게 들리지만, 결국은 여전히 벡터 기반/인코더 기반 관련 컨텍스트 검색이고, 그래프 DB를 쓰든 뭘 쓰든 결국 "검색으로 생성을 보강하는(retrieval-augmented generation)" 기술을 쓰고 있는 것 → 그러니 여전히 RAG.

RAG는 죽지 않았고 이름이 바뀌어도 본질은 여전히 RAG라는 결론.




In [62]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import os
from langchain_text_splitters import MarkdownTextSplitter
from langchain_core.documents import Document

load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')

load_dotenv(override=True)

MODEL = "gemini-3.1-flash-lite"

DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

ollama_url = "http://localhost:11434/v1"
ollama_api = OpenAI(api_key="ollama", base_url=ollama_url)
OLLAMA_MODEL = "gemma4:e4b"


In [3]:
# LangChain의 Document 클래스에서 영감을 받아 - 비슷한 걸 만들어봅시다

class Result(BaseModel):
    page_content: str
    metadata: dict

In [4]:
class Chunk(BaseModel):
    headline: str = Field(description="이 청크에 대한 짧은 제목 (보통 몇 단어), 쿼리에서 노출될 가능성이 가장 높은 제목")
    summary: str = Field(description="흔히 나올 법한 질문에 답할 수 있도록 이 청크의 내용을 몇 문장으로 요약한 것")
    original_text: str = Field(description="제공된 문서에서 가져온 이 청크의 원본 텍스트, 어떤 변경도 없이 정확히 그대로")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

## 세 단계:

1. LangChain에서 했던 것처럼 지식 베이스(knowledge base)에서 문서를 가져옵니다
2. LLM을 호출해서 문서를 청크(Chunk)로 변환합니다
3. 청크를 Chroma에 저장합니다

이게 전부입니다!

In [5]:
# 1단계 
def fetch_documents():
    """LangChain의 DirectoryLoader를 직접 만들어본 버전"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [6]:
documents = fetch_documents()

Loaded 76 documents


In [7]:
# 2단계
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
당신은 문서를 받아서 지식 베이스(KnowledgeBase)를 위해 겹치는 청크(chunk)들로 나눕니다.

이 문서는 Insurellm이라는 회사의 공유 드라이브에서 가져온 것입니다.
문서의 종류: {document["type"]}
문서를 가져온 출처: {document["source"]}

챗봇이 이 청크들을 사용해서 회사에 대한 질문에 답할 것입니다.
문서 전체가 청크들에 빠짐없이 포함되도록, 적절하다고 생각하는 방식으로 문서를 나누세요 - 아무것도 빠뜨리지 마세요.
이 문서는 아마 {how_many}개의 청크로 나누는 것이 적당하겠지만, 필요에 따라 더 많거나 적어도 됩니다.
청크들 사이에는 적절한 중복(overlap)이 있어야 합니다. 보통 약 25%의 중복, 또는 약 50단어 정도로, 최선의 검색 결과를 위해 여러 청크에 동일한 텍스트가 포함되도록 하세요.

각 청크마다 제목(headline), 요약(summary), 그리고 청크의 원본 텍스트를 제공해야 합니다.
청크들을 모두 합치면 중복을 포함해 문서 전체를 나타내야 합니다.

다음은 문서입니다:

{document["text"]}

청크들로 응답하세요.
"""

In [ ]:
print(make_prompt(documents[0]))

In [9]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [10]:
make_messages(documents[0])

[{'role': 'user',
  'content': '\n당신은 문서를 받아서 지식 베이스(KnowledgeBase)를 위해 겹치는 청크(chunk)들로 나눕니다.\n\n이 문서는 Insurellm이라는 회사의 공유 드라이브에서 가져온 것입니다.\n문서의 종류: company\n문서를 가져온 출처: knowledge-base/company/about.md\n\n챗봇이 이 청크들을 사용해서 회사에 대한 질문에 답할 것입니다.\n문서 전체가 청크들에 빠짐없이 포함되도록, 적절하다고 생각하는 방식으로 문서를 나누세요 - 아무것도 빠뜨리지 마세요.\n이 문서는 아마 3개의 청크로 나누는 것이 적당하겠지만, 필요에 따라 더 많거나 적어도 됩니다.\n청크들 사이에는 적절한 중복(overlap)이 있어야 합니다. 보통 약 25%의 중복, 또는 약 50단어 정도로, 최선의 검색 결과를 위해 여러 청크에 동일한 텍스트가 포함되도록 하세요.\n\n각 청크마다 제목(headline), 요약(summary), 그리고 청크의 원본 텍스트를 제공해야 합니다.\n청크들을 모두 합치면 중복을 포함해 문서 전체를 나타내야 합니다.\n\n다음은 문서입니다:\n\n# Insurellm 소개\n\nInsurellm은 혁신적인 제품을 필요로 하는 업계에 변화를 일으키기 위한 보험 기술 스타트업으로서 2015년 Avery Lancaster에 의해 설립되었습니다. 첫 번째 제품은 소비자와 보험사를 연결하는 마켓플레이스인 Markellm이었습니다.\n\n회사는 설립 후 첫 5년 동안 빠르게 성장하며 제품 포트폴리오를 확장하여 Carllm(자동차 보험 포털), Homellm(주택 보험 포털), Rellm(기업용 재보험 플랫폼)을 추가했습니다. 2020년까지 Insurellm은 미국 전역 12개 사무소에 200명의 직원을 두는 정점에 도달했습니다.\n\n그러나 회사는 수익성과 지속 가능한 성장에 집중하기 위해 2022~2023년에 전략적 구조조정을 단행했습니다. 여기에는 사무소 위치 통합, 원격 근무 

In [28]:
# def process_document(document):
#     messages = make_messages(document)
#     response = completion(model="gemini/gemini-3.1-flash-lite", messages=messages, response_format=Chunks)
#     reply = response.choices[0].message.content
#     doc_as_chunks = Chunks.model_validate_json(reply).chunks
#     return [chunk.as_result(document) for chunk in doc_as_chunks]


# Ollama 
def process_document(document):
    messages = make_messages(document)
    response = completion(
        model="ollama_chat/qwen2.5:3b-instruct-q4_K_M",  # 사용 중인 로컬 모델명으로 교체
        messages=messages,
        response_format=Chunks,
        api_base="http://localhost:11434",  # 원격/다른 포트면 수정
    )
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]


In [29]:
process_document(documents[0])

[Result(page_content='Insurellm의 역사 및 성장\n\nInsurellm은 2015년 설립된 보험 기술 스타트업으로, 2020년까지 전역 12개 사무소와 200명의 직원을 확장했습니다.\n\nInsurellm은 혁신적인 제품을 필요로 하는 업계에 변화를 일으키기 위한 보험 기술 스타트업으로서 2015년 Avery Lancaster에 의해 설립되었습니다. 첫 번째 제품은 소비자와 보험사를 연결하는 마켓플레이스인 Markellm이었습니다. 회사는 설립 후 첫 5년 동안 빠르게 성장하며 제품 포트폴리오를 확장하여 Carllm(자동차 보험 포털), Homellm(주택 보험 포털), Rellm(기업용 재보험 플랫폼)을 추가했습니다. 2020년까지 Insurellm은 미국 전역 12개 사무소에 200명의 직원을 두는 정점에 도달했습니다. 그으 날 회사는 2022~2023년에 수익성과 지속 가능한 성장에 집중하기 위해 전략적 구조조정을 단행했습니다. 여기에는 사무소 위치 통합, 원격 근무 우선 전략 도입, 운영 간소화가 포함되었습니다. 2025년 현재 Insurellm은 32명의 정예 인력으로 구성된 효율적인 팀을 운영하고 있으며, 8개 전 제품 라인에 걸쳐 32건의 활성 계약으로 이루어진 포트폴리오를 구축했습니다. 회사는 샌프란시스코 본사와 함께 뉴욕, 오스틴, 시카고, 덴버 등 주요 시장에 소규모 위성 사무소를 유지하고 있습니다. 구조조정 이후에도 Insurellm은 혁신을 지속하여 제품군을 8개의 종합 플랫폼으로 확장했습니다. 회사는 보험 수요의 전 영역을 아우르기 위해 Lifellm(생명보험), Healthllm(건강보험), Bizllm(기업보험), Claimllm(보험금 청구 처리)을 추가했습니다. 이러한 전략적 확장은 모든 신규 제품에서 강력한 채택률을 보이며 매우 성공적이었습니다. - **Bizllm**은 지역 보험사와 전국 규모의 기업보험 그룹을 포함하여 7건의 기업보험 계약을 신속하게 확보했습니다 - **Claimllm**은 독립 손

In [49]:
# 여기서 한 문서마다 한번의 API를 호출한다.
# 문서 한개를 청크로 만드는데 API를 사용한다.
# def create_chunks(documents):
#     chunks = []
#     for doc in tqdm(documents):
#         chunks.extend(process_document(doc))
#     return chunks

# ollama는 이 작업하기가 부담스러움
# def create_chunks(documents):
#     chunks = []
#     for doc in tqdm(documents[:6]):
#         chunks.extend(process_document(doc))
#     return chunks

# 그냥 랭체인 사용
def create_chunks(documents):
    docs = [
        Document(
            page_content=d["text"],
            metadata={"type": d["type"], "source": d["source"]},
        )
        for d in documents
    ]
    text_splitter = MarkdownTextSplitter(chunk_size=500, chunk_overlap=200)
    chunks = text_splitter.split_documents(docs)
    return chunks

In [50]:
chunks = create_chunks(documents)
print(len(chunks))

543


In [65]:
# 3단계 : 임베딩 저장하기

# def create_embeddings(chunks):
#     chroma = PersistentClient(path=DB_NAME)
#     if collection_name in [c.name for c in chroma.list_collections()]:
#         chroma.delete_collection(collection_name)

#     texts = [chunk.page_content for chunk in chunks]
#     # 아마 안될꺼임
#     emb = gemini.embeddings.create(model=embedding_model, input=texts).data
#     vectors = [e.embedding for e in emb]

#     collection = chroma.get_or_create_collection(collection_name)

#     ids = [str(i) for i in range(len(chunks))]
#     metas = [chunk.metadata for chunk in chunks]

#     collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
#     print(f"Vectorstore created with {collection.count()} documents")

import ollama as ollama_client
client = ollama_client.Client(host="http://localhost:11434")

def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]

    vectors = [
        client.embed(model="bge-m3", input=text)["embeddings"][0]
        for text in texts
    ]

    collection = chroma.get_or_create_collection(collection_name)
    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [66]:
create_embeddings(chunks)

Vectorstore created with 543 documents


In [ ]:
## 1. Reranking (재순위화) - 검색 결과의 순위를 재조정

In [68]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="청크 id 번호를 기준으로, 관련성이 가장 높은 것부터 가장 낮은 것까지 정렬한 순서"
    )

In [69]:
def rerank(question, chunks):
    system_prompt = """
당신은 문서 재순위화(re-ranker) 담당자입니다.
질문 하나와, 지식 베이스 쿼리에서 검색된 관련 텍스트 청크들의 목록이 주어집니다.
청크들은 검색된 순서대로 제공되며, 이는 대략적으로 관련성 순으로 정렬되어 있지만 당신이 이를 더 개선할 수 있을 것입니다.
제공된 청크들을 질문과의 관련성에 따라 순위를 매겨야 하며, 가장 관련성이 높은 청크가 맨 앞에 오도록 하세요.
순위가 매겨진 청크 id 목록만 응답하고, 그 외의 것은 아무것도 답하지 마세요. 제공받은 모든 청크 id를 순위를 매겨 포함시키세요.
"""
    user_prompt = f"사용자가 다음과 같은 질문을 했습니다:\n\n{question}\n\n질문과의 관련성에 따라 모든 텍스트 청크의 순서를 가장 관련성이 높은 것부터 낮은 것 순으로 정렬하세요. 제공받은 모든 청크 id를 순위를 매겨 포함시키세요.\n\n"
    user_prompt += "다음은 청크들입니다:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "순위가 매겨진 청크 id 목록만 응답하고, 그 외의 것은 아무것도 답하지 마세요."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model="gemini/gemini-3.1-flash-lite", messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [72]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)

In [73]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    # 변경해야함
    query = client.embed(model="bge-m3", input=[question])["embeddings"][0]
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [74]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

In [75]:
chunks

[Result(page_content='## 기타 인사 노트\n- Maxine은 빅데이터 기술 및 클라우드 인프라 관련 다양한 회사 후원 교육에 참여했습니다.\n- 2023년 권위 있는 Insurellm IIOTY Innovator Award를 수상하며 그녀의 기여를 인정받았습니다.\n- Maxine은 현재 여성 IT 인재 지원 이니셔티브에 참여하고 있으며 주니어 직원들을 지도하는 멘토십 프로그램에도 참여하고 있습니다.\n- 향후 개발 영역으로는 프로젝트 전환과 협업을 원활하게 하기 위한 이해관계자 커뮤니케이션 능력 개선이 있습니다.', metadata={'type': 'employees', 'source': 'knowledge-base/employees/Maxine Thompson.md'}),
 Result(page_content="## 기타 인사 노트\n- **학력:** 일리노이 대학교 경영학 학사\n- **수상:** 2021년 Sales Excellence Award 수상, 2019년, 2021년 President's Club 멤버\n- **파이프라인 관리:** 부진했던 2023년 이후 파이프라인 재구축을 위해 노력 중. 매니저가 추가 지원 및 마케팅 리소스 제공\n- **역량:** 엔터프라이즈 관계 구축 및 복잡한 딜 협상에 강점. 긴 영업 사이클 경험 보유\n- **개발 초점:** 계정 기반 영업 전략과 소셜 셀링 기법 활용 방안을 익히는 중\n- **피드백:** 깊은 업계 지식을 갖춘 노련한 영업 전문가. 대형 전략 딜에서 최고의 성과를 발휘함. 일관된 잠재고객 발굴 규율을 유지할 필요가 있음", metadata={'source': "knowledge-base/employees/Michael O'Brien.md", 'type': 'employees'}),
 Result(page_content='## 기타 인사 노트\n- **학력:** 텍사스 대학교 오스틴 캠퍼스 마케팅 MBA, 커뮤니케이션학 학사\n- **자격증:** Google Analytics 

In [76]:
for chunk in chunks:
    print(chunk.page_content[:15]+"...")

## 기타 인사 노트
- M...
## 기타 인사 노트
- *...
## 기타 인사 노트
- *...
## 연간 성과 이력
- *...
---

**서명:**

_...
## 연간 성과 이력
- *...
- **2022년:** 평가...
---

**서명:**

_...
## 기타 인사 참고사항
-...
## 기타 인사 노트
- *...


In [77]:
reranked = rerank(question, chunks)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [78]:
for chunk in reranked:
    print(chunk.page_content[:15]+"...")

## 기타 인사 노트
- M...
## 기타 인사 노트
- *...
## 기타 인사 노트
- *...
## 연간 성과 이력
- *...
---

**서명:**

_...
## 연간 성과 이력
- *...
- **2022년:** 평가...
---

**서명:**

_...
## 기타 인사 참고사항
-...
## 기타 인사 노트
- *...


In [80]:
question = "Who went to Manchester University?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

In [81]:
chunks

[Result(page_content='## 기타 인사 노트\n- **학력:** 맨체스터 대학교 컴퓨터공학 학사\n- **역량:** React, TypeScript, HTML/CSS, 테스트를 위한 Jest에 능숙. Next.js와 GraphQL 학습 중\n- **전문성 개발:** Advanced React Patterns 과정 수료 (2023년). 오픈소스 프로젝트에 적극적으로 기여\n- **업무 스타일:** 원격 근무를 선호함. 뛰어난 문서 커뮤니케이션 능력. 팀 스탠드업과 계획 세션에 적극적으로 참여\n- **피드백:** UI 디테일에 대한 세심함을 갖춘 신뢰할 수 있는 개발자. 기술적 의사결정 역량이 향상되고 있음. 더 큰 규모의 기능을 주도적으로 맡으면 도움이 될 것', metadata={'source': 'knowledge-base/employees/Jessica Liu.md', 'type': 'employees'}),
 Result(page_content='## 기타 인사 노트\n- **학력:** 텍사스 대학교 오스틴 캠퍼스 마케팅 MBA, 커뮤니케이션학 학사\n- **자격증:** Google Analytics 공인 자격증, HubSpot 마케팅 자동화 전문가\n- **수상:** 2023년 Marketing Excellence Award 수상, 수상 경력이 있는 리브랜딩 캠페인 주도\n- **팀 리더십:** 마케팅 스페셜리스트 5명으로 구성된 팀 관리. 인재 육성과 고성과 문화 조성으로 정평이 나 있음\n- **역량:** 수요 창출, 콘텐츠 마케팅, 마케팅 분석, 마케팅 자동화(HubSpot, Marketo), 브랜드 전략에 전문성 보유\n- **피드백:** 강력한 분석력과 창의적 비전을 갖춘 전략적 마케팅 리더. 측정 가능한 비즈니스 성과를 이끌어내는 뛰어난 교차 기능 협업자', metadata={'type': 'employees', 'source': 'knowledge-base/employees/Lisa Anderson.md'}),
 Resul

In [82]:
reranked = rerank(question, chunks)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


In [85]:
for index, c in enumerate(reranked):
    if "맨체스터" in c.page_content.lower():
        print(index)

0


In [86]:
reranked[0].page_content

'## 기타 인사 노트\n- **학력:** 맨체스터 대학교 컴퓨터공학 학사\n- **역량:** React, TypeScript, HTML/CSS, 테스트를 위한 Jest에 능숙. Next.js와 GraphQL 학습 중\n- **전문성 개발:** Advanced React Patterns 과정 수료 (2023년). 오픈소스 프로젝트에 적극적으로 기여\n- **업무 스타일:** 원격 근무를 선호함. 뛰어난 문서 커뮤니케이션 능력. 팀 스탠드업과 계획 세션에 적극적으로 참여\n- **피드백:** UI 디테일에 대한 세심함을 갖춘 신뢰할 수 있는 개발자. 기술적 의사결정 역량이 향상되고 있음. 더 큰 규모의 기능을 주도적으로 맡으면 도움이 될 것'

In [ ]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [ ]:
# 쿼리 재작성 (Query re-writing)

SYSTEM_PROMPT = """
당신은 Insurellm이라는 회사를 대표하는, 지식이 풍부하고 친절한 어시스턴트입니다.
당신은 Insurellm에 대해 사용자와 대화하고 있습니다.
당신의 답변은 정확성, 관련성, 완전성 측면에서 평가될 것이므로, 질문에만 답하되 완전하게 답하도록 하세요.
답을 모른다면, 모른다고 말하세요.
참고할 수 있도록, 사용자의 질문과 직접적으로 관련이 있을 수 있는 지식 베이스의 특정 발췌 내용은 다음과 같습니다:
{context}

이 맥락을 바탕으로 사용자의 질문에 답변해 주세요. 정확하고, 관련성 있으며, 완전하게 답변하세요.
"""

In [ ]:
def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"{chunk.metadata['source']}에서 발췌:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [ ]:
def rewrite_query(question, history=[]):
    """사용자의 질문을 지식 베이스에서 관련 콘텐츠를 더 잘 찾아낼 수 있도록 더 구체적인 질문으로 다시 작성한다."""
    message = f"""
당신은 사용자와 대화하며 Insurellm이라는 회사에 대한 질문에 답하고 있습니다.
사용자의 질문에 답하기 위해 지식 베이스(Knowledge Base)에서 정보를 찾아보려는 참입니다.

지금까지 사용자와 나눈 대화 기록은 다음과 같습니다:
{history}

그리고 사용자의 현재 질문은 다음과 같습니다:
{question}

지식 베이스를 검색하는 데 사용할, 다듬어진 질문 하나만 응답하세요.
관련 콘텐츠를 찾아낼 가능성이 가장 높은 매우 짧고 구체적인 질문이어야 합니다. 질문의 세부 사항에 집중하세요.
회사에 대한 일반적인 질문이 아니라면 회사 이름은 언급하지 마세요.
중요: 지식 베이스 쿼리만 응답하고, 그 외의 것은 아무것도 답하지 마세요.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [ ]:
rewrite_query("Who won the IIOTY award?", [])

In [ ]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    RAG를 사용해서 질문에 답하고, 답변과 검색된 context를 반환한다
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [ ]:
answer_question("Who won the IIOTY award?", [])

In [ ]:
answer_question("Who went to Manchester University?", [])